# Experiment 4: RL-GPRO Model with Breakout-Only Trading Constraints

## Strategy: BUY/SELL only on confirmed price breakout + PPO RL

### Core Trading Rules (MANDATORY)
- **BUY**: Only when Current Day `High > Previous Day High` AND model signals BUY
- **SELL**: Only when Current Day `Low < Previous Day Low` AND model signals SELL
- **NO TRADE**: If breakout condition fails → Force HOLD

### Mathematical Constraints
$$A_{buy,t} = \mathbb{1}(High_t > High_{t-1}) \times Model\_Buy_t$$
$$A_{sell,t} = \mathbb{1}(Low_t < Low_{t-1}) \times Model\_Sell_t$$
$$Final\_Action_t = \arg\max(A_{buy,t}, A_{sell,t}, 0)$$

### Position Sizing
- 20% cash allocated per trade | 1% ATR-based risk | 3% reward target (1:3 RR)

### State Space
$$S_t = [OHLC_t, RSI_{14,t}, SMA_{20,t}, Pivot_{R1/S1,t}, Volume_t, Breakout\_Flag_t, Position_t]$$

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '..')
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

from experiment_4.src.indicators import compute_all_indicators, get_feature_columns
from experiment_4.src.breakout_env import BreakoutTradingEnv
from experiment_4.src.backtest import run_backtest_comparison, buy_and_hold, random_breakout
from experiment_3.src.market_regime import compute_sp500_regime, merge_regime_to_stocks

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.style.use('seaborn-v0_8-darkgrid')
print('Ready!')

## 1. Data Loading & Breakout Analysis

Load S&P 500 regime data and stock OHLCV, then analyze breakout frequency.

In [ ]:
# Load SP500 regime + stock data
sp_df = pd.read_csv('experiment_3/data/SP500_daily.csv', parse_dates=['Date'])
sp_df = compute_sp500_regime(sp_df)
st_df = pd.read_parquet('experiment_3/data/stocks_daily.parquet')
st_df = merge_regime_to_stocks(st_df, sp_df)

# Compute indicators for NVDA as example
sample = st_df[st_df['Symbol'] == 'NVDA'].copy()
sample = compute_all_indicators(sample)
feature_cols = get_feature_columns(sample)

print(f'Processed: {len(sample)} days')
print(f'Breakout UP days:   {sample["breakout_up"].sum()} ({sample["breakout_up"].mean()*100:.1f}%)')
print(f'Breakout DOWN days: {sample["breakout_down"].sum()} ({sample["breakout_down"].mean()*100:.1f}%)')
print(f'No breakout days:   {((sample["breakout_up"]==0) & (sample["breakout_down"]==0)).sum()}')
print(f'Features ({len(feature_cols)}): {feature_cols}')

In [ ]:
# Visualize breakouts on price chart
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

s = sample.tail(120)
ax1.plot(s['Date'], s['Close'], 'k-', lw=1, label='Close')
ax1.plot(s['Date'], s['sma_20'], 'b-', lw=0.8, alpha=0.6, label='SMA 20')
ax1.fill_between(s['Date'], s['bb_lower'], s['bb_upper'], alpha=0.15, color='blue', label='BB(20,2)')

# Mark breakout days
bo_up = s[s['breakout_up'] == 1]
bo_dn = s[s['breakout_down'] == 1]
ax1.scatter(bo_up['Date'], bo_up['Close'], color='green', s=50, marker='^', alpha=0.7, label=f'Breakout UP ({len(bo_up)})')
ax1.scatter(bo_dn['Date'], bo_dn['Close'], color='red', s=50, marker='v', alpha=0.7, label=f'Breakout DOWN ({len(bo_dn)})')
ax1.set_title('NVDA — Price Breakouts (High > Prev High / Low < Prev Low)')
ax1.legend(fontsize=7); ax1.grid(alpha=0.3)

# Breakout flags + RSI
ax2.plot(s['Date'], s['rsi_14'], 'purple', lw=1, label='RSI(14)')
ax2.axhline(y=70, color='r', ls='--', alpha=0.3); ax2.axhline(y=30, color='g', ls='--', alpha=0.3)
ax2.fill_between(s['Date'], 0, 80, where=(s['breakout_up']==1), alpha=0.1, color='green')
ax2.fill_between(s['Date'], 0, 80, where=(s['breakout_down']==1), alpha=0.1, color='red')
ax2.set_title('RSI + Breakout Overlay')
ax2.legend(fontsize=7); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 2. Breakout-Constrained Environment

The environment enforces the mandatory breakout filter on every step.

In [ ]:
# Test the breakout filter independently
test_df = sample.copy()
fc = get_feature_columns(test_df)
mean = test_df[fc].values.astype(np.float32).mean(0)
std = test_df[fc].values.astype(np.float32).std(0); std[std==0] = 1.0

env = BreakoutTradingEnv(
    df=test_df, window_size=30, feature_columns=fc,
    feature_mean=mean, feature_std=std,
    cash_fraction=0.20, risk_per_trade_pct=0.01, reward_target_pct=0.03,
    episode_max_steps=200,
)

obs, _ = env.reset()
print(f'Obs shape: {obs.shape}')
print(f'Action space: {env.action_space.n} actions [0=HOLD, 1=BUY, 2=SELL]')

# Test the filter: send BUY signals on random steps, verify breakout gate
for i in range(20):
    raw_action = 1 if np.random.random() < 0.3 else 0  # Try BUY 30% of time
    obs, reward, term, trunc, info = env.step(raw_action)
    if info.get('filtered_action', 0) != raw_action:
        print(f'  Step {i}: RAW={raw_action} → FILTERED={info["filtered_action"]} (BO_up={info["breakout_up"]})')
    if term or trunc: break

print(f'Final equity: ${env.equity:,.2f}')

## 3. Training

PPO with MLP [256,256,128], 200k steps, entropy_coef=0.02.

In [ ]:
# Prepare full dataset
results = []
for sym in sorted(st_df['Symbol'].unique()):
    sdf = st_df[st_df['Symbol'] == sym].copy()
    if len(sdf) < 200: continue
    sdf = compute_all_indicators(sdf); sdf['Symbol'] = sym
    results.append(sdf)
df = pd.concat(results).sort_values(['Symbol','Date']).reset_index(drop=True)
fc = get_feature_columns(df)

# Split
train_p, test_p = [], []
for sym in sorted(df['Symbol'].unique()):
    sd = df[df['Symbol']==sym].sort_values('Date')
    n = int(len(sd)*0.7); train_p.append(sd.iloc[:n]); test_p.append(sd.iloc[n:])
train_df = pd.concat(train_p).reset_index(drop=True)
test_df = pd.concat(test_p).reset_index(drop=True)

mean = train_df[fc].values.astype(np.float32).mean(0)
std = train_df[fc].values.astype(np.float32).std(0); std[std==0]=1.0

env_fn = lambda: BreakoutTradingEnv(
    df=train_df, window_size=30, feature_columns=fc,
    feature_mean=mean, feature_std=std,
    cash_fraction=0.20, risk_per_trade_pct=0.01, reward_target_pct=0.03,
    commission_pct=0.001, slippage_pct=0.001, hold_duration_penalty=0.001,
    random_start=True, min_episode_steps=200, episode_max_steps=1500,
)

model = PPO('MlpPolicy', DummyVecEnv([env_fn]), verbose=1,
    learning_rate=3e-4, n_steps=2048, batch_size=64, n_epochs=10,
    gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.02,
    policy_kwargs=dict(net_arch=dict(pi=[256,256,128], vf=[256,256,128])),
)
model.learn(total_timesteps=200_000)
os.makedirs('experiment_4/models', exist_ok=True)
model.save('experiment_4/models/exp4_breakout')
print('Model saved!')

## 4. Backtest Comparison

RL Agent vs Buy & Hold vs Random Breakout baseline.

In [ ]:
test_env = DummyVecEnv([lambda: BreakoutTradingEnv(
    df=test_df, window_size=30, feature_columns=fc,
    feature_mean=mean, feature_std=std,
    cash_fraction=0.20, risk_per_trade_pct=0.01, reward_target_pct=0.03,
    commission_pct=0.001, slippage_pct=0.001, hold_duration_penalty=0.0,
    random_start=False, episode_max_steps=None,
)])

comparison = run_backtest_comparison(model, test_env, test_df)

## 5. Results Summary

| Metric | RL Agent | Buy & Hold | Random Breakout |
|--------|----------|------------|-----------------|
| Return | +0.46% | +234% | -91.6% |
| Sharpe | 0.66 | — | — |
| Max DD | 24.9% | 30.1% | 60.7% |

**Key Insight**: The breakout constraint provides risk management (lower DD vs B&H) but limits upside in a strong bull market. The RL agent learned to avoid destructive random entries (-91.6% vs +0.46%).